# Notebook 02 — Phase 1: Pretrained Models → TFLite

**Phase:** 1 (Real-but-weak models)  
**Purpose:** Replace three of the four dummy `.tflite` files in `mobile_app/assets/models/` with **real pretrained weights**, so the mobile JSI bridge starts producing meaningful outputs instead of random vectors.  
**Expected runtime:** ~5–10 minutes (mostly downloads)  
**GPU required:** No (CPU runtime is fine — save GPU quota for Notebook 03)

## What we ship after this notebook

| Model | Source | Status after this notebook |
| --- | --- | --- |
| BlazeFace (detection) | MediaPipe GCS, pretrained | **Real** |
| FaceMesh (landmarks for EAR/MAR/PnP) | MediaPipe GCS, pretrained | **Real** |
| MobileFaceNet (identity, 512-D) | InsightFace `buffalo_s/w600k_mbf.onnx` → onnx2tf → `.tflite` | **Real** (pretrained on WebFace600K, no fine-tune yet) |
| ShuffleNet (liveness) | — | Still dummy (trained in Notebook 03) |

These models match the shapes in `shared_contracts/README.md`. The mobile side does not need to change code — only swap the filenames it loads.

## What to send back to Claude

After running, paste the **full text output of the final summary cell** (cell 13). If anything fails, paste the **full traceback** of the failing cell.

## After it runs (what you do locally)

1. Download the four `.tflite` files from Kaggle Output → `/kaggle/working/models/`.
2. Move them into `mobile_app/assets/models/` on your laptop (these are NEW filenames — don't delete the dummies yet, keep them as fallback for one commit).
3. Commit with a clear message; ledger entry.
4. Tell teammate: "point the mobile loader at `blazeface.tflite`, `facemesh.tflite`, `mobilefacenet.tflite` — the dummies stay as fallback for now."

## 1 — Clone the repo (same scaffold as Notebook 01)

In [ ]:
import os, subprocess

WORK_DIR = "/kaggle/working"
REPO_URL = "https://github.com/MalayM09/nhai.git"
REPO_DIR = os.path.join(WORK_DIR, "nhai")

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("CWD:", os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)

## 2 — Install conversion dependencies

- `onnx2tf` (PINTO0309) — converts ONNX (NCHW) → TFLite (NHWC) with sane defaults.
- `onnx` + `onnxruntime` — needed by onnx2tf and for sanity-checking the ONNX before conversion.
- `tf-keras` — Keras 3 compat layer for older Keras code paths.

Kaggle already has TensorFlow and numpy.

In [ ]:
!pip install -q onnx2tf onnx onnxruntime onnx_graphsurgeon sng4onnx tf-keras
# create the output staging dir up front
STAGING = "/kaggle/working/models"
os.makedirs(STAGING, exist_ok=True)
print("Staging dir:", STAGING)

## 3 — MediaPipe BlazeFace (short-range)

Pretrained, no training needed. Two-source download with fallback in case Google rotates GCS paths.

In [ ]:
import urllib.request, hashlib

def download_with_fallback(urls, dest):
    last_err = None
    for url in urls:
        try:
            print(f"  GET {url}")
            urllib.request.urlretrieve(url, dest)
            size = os.path.getsize(dest)
            assert size > 1000, f"file too small ({size} bytes), likely an HTML error page"
            print(f"  OK  {dest}  ({size/1024:.1f} KB)")
            return
        except Exception as e:
            print(f"  FAIL ({type(e).__name__}: {e})")
            last_err = e
    raise RuntimeError(f"all sources failed; last error: {last_err}")

BLAZEFACE_DEST = os.path.join(STAGING, "blazeface.tflite")
BLAZEFACE_SOURCES = [
    "https://storage.googleapis.com/mediapipe-assets/face_detection_short_range.tflite",
    "https://github.com/google-ai-edge/mediapipe/raw/master/mediapipe/modules/face_detection/face_detection_short_range.tflite",
    "https://github.com/google/mediapipe/raw/master/mediapipe/modules/face_detection/face_detection_short_range.tflite",
]
download_with_fallback(BLAZEFACE_SOURCES, BLAZEFACE_DEST)

In [ ]:
# Sanity check BlazeFace shape
import tensorflow as tf

it = tf.lite.Interpreter(model_path=BLAZEFACE_DEST)
it.allocate_tensors()
print("BlazeFace inputs:")
for t in it.get_input_details():
    print(f"  {t['name']:30s} shape={t['shape'].tolist()} dtype={t['dtype'].__name__}")
print("BlazeFace outputs:")
for t in it.get_output_details():
    print(f"  {t['name']:30s} shape={t['shape'].tolist()} dtype={t['dtype'].__name__}")

# Contract: input [1,128,128,3]; outputs cover boxes [1,896,16] and scores [1,896,1].
# The exact MediaPipe variant may emit slightly different output names but the same anchor count.
input_shape = it.get_input_details()[0]['shape'].tolist()
assert input_shape == [1, 128, 128, 3], f"BlazeFace input shape {input_shape} != [1, 128, 128, 3] — verify URL"

## 4 — MediaPipe FaceMesh (468 landmarks)

Used mobile-side for EAR / MAR / PnP heuristics in Gate 1. Pretrained, no training needed.

In [ ]:
FACEMESH_DEST = os.path.join(STAGING, "facemesh.tflite")
FACEMESH_SOURCES = [
    "https://storage.googleapis.com/mediapipe-assets/face_landmark.tflite",
    "https://github.com/google-ai-edge/mediapipe/raw/master/mediapipe/modules/face_landmark/face_landmark.tflite",
    "https://github.com/google/mediapipe/raw/master/mediapipe/modules/face_landmark/face_landmark.tflite",
]
download_with_fallback(FACEMESH_SOURCES, FACEMESH_DEST)

In [ ]:
# Sanity check FaceMesh shape
it = tf.lite.Interpreter(model_path=FACEMESH_DEST)
it.allocate_tensors()
print("FaceMesh inputs:")
for t in it.get_input_details():
    print(f"  {t['name']:30s} shape={t['shape'].tolist()} dtype={t['dtype'].__name__}")
print("FaceMesh outputs:")
for t in it.get_output_details():
    print(f"  {t['name']:30s} shape={t['shape'].tolist()} dtype={t['dtype'].__name__}")

# Contract: input [1,192,192,3]; output flattens to 468 landmarks * 3 coords = 1404 floats.
input_shape = it.get_input_details()[0]['shape'].tolist()
assert input_shape == [1, 192, 192, 3], f"FaceMesh input shape {input_shape} != [1, 192, 192, 3] — verify URL"

## 5 — MobileFaceNet via InsightFace ONNX → TFLite

We pull the **`w600k_mbf.onnx`** model from InsightFace's `buffalo_s` model pack. This is genuine MobileFaceNet trained on WebFace600K — exactly what the brief specifies, and what judges will recognize.

Steps:
1. Download `buffalo_s.zip` from InsightFace's GitHub release.
2. Extract `w600k_mbf.onnx` (~13 MB ONNX file).
3. Convert ONNX (NCHW, `[1, 3, 112, 112]`) → TFLite FP32 (NHWC, `[1, 112, 112, 3]`) using `onnx2tf`.
4. Verify the output tensor is `[1, 512]`.

> **Preprocessing note for mobile:** InsightFace MobileFaceNet expects pixels normalized to `[-1, 1]` (specifically `(pixel - 127.5) / 127.5`). This matches our `shared_contracts/README.md`. The output `[1, 512]` is **NOT** pre-L2-normalized — the mobile side must L2-normalize before computing cosine distance, per the contract.

In [ ]:
import zipfile, shutil

BUFFALO_ZIP = "/kaggle/working/buffalo_s.zip"
BUFFALO_DIR = "/kaggle/working/buffalo_s"
BUFFALO_SOURCES = [
    "https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_s.zip",
]

download_with_fallback(BUFFALO_SOURCES, BUFFALO_ZIP)

os.makedirs(BUFFALO_DIR, exist_ok=True)
with zipfile.ZipFile(BUFFALO_ZIP) as z:
    z.extractall(BUFFALO_DIR)

print("Extracted:")
for root, _, files in os.walk(BUFFALO_DIR):
    for f in files:
        p = os.path.join(root, f)
        print(f"  {p}  ({os.path.getsize(p)/1024/1024:.2f} MB)")

# locate the MobileFaceNet ONNX
ONNX_PATH = None
for root, _, files in os.walk(BUFFALO_DIR):
    for f in files:
        if f == "w600k_mbf.onnx":
            ONNX_PATH = os.path.join(root, f)
assert ONNX_PATH is not None, "w600k_mbf.onnx not found in buffalo_s — InsightFace may have renamed the file"
print(f"\nMobileFaceNet ONNX: {ONNX_PATH}")

In [ ]:
# Inspect the ONNX before conversion so we know exactly what onnx2tf is working with
import onnx

model = onnx.load(ONNX_PATH)
print("ONNX inputs:")
for x in model.graph.input:
    dims = [d.dim_value if d.dim_value else d.dim_param for d in x.type.tensor_type.shape.dim]
    print(f"  {x.name}: {dims}")
print("ONNX outputs:")
for x in model.graph.output:
    dims = [d.dim_value if d.dim_value else d.dim_param for d in x.type.tensor_type.shape.dim]
    print(f"  {x.name}: {dims}")
print(f"\nopset: {[o.version for o in model.opset_import]}")

In [ ]:
# Convert ONNX -> TFLite via onnx2tf. By default it transposes NCHW -> NHWC for image inputs.
MBF_OUT_DIR = "/kaggle/working/mobilefacenet_tflite"
shutil.rmtree(MBF_OUT_DIR, ignore_errors=True)
os.makedirs(MBF_OUT_DIR, exist_ok=True)

# -b 1 fixes batch to 1 (matches our contract). onnx2tf emits a directory of variants.
!onnx2tf -i {ONNX_PATH} -o {MBF_OUT_DIR} -b 1 -ois input.1:1,3,112,112

# List what onnx2tf produced
print("\nonnx2tf outputs:")
for f in sorted(os.listdir(MBF_OUT_DIR)):
    p = os.path.join(MBF_OUT_DIR, f)
    print(f"  {f}  ({os.path.getsize(p)/1024/1024:.2f} MB)")

In [ ]:
# Pick the FP32 variant for Phase 1 (INT8 PTQ comes in Notebook 05).
# onnx2tf naming is usually: <stem>_float32.tflite
candidates = [f for f in os.listdir(MBF_OUT_DIR) if f.endswith(".tflite") and "float32" in f]
if not candidates:
    # Fallback: any .tflite
    candidates = [f for f in os.listdir(MBF_OUT_DIR) if f.endswith(".tflite")]
assert candidates, f"no .tflite emitted in {MBF_OUT_DIR}"

MBF_TFLITE_SRC = os.path.join(MBF_OUT_DIR, candidates[0])
MOBILEFACENET_DEST = os.path.join(STAGING, "mobilefacenet.tflite")
shutil.copy(MBF_TFLITE_SRC, MOBILEFACENET_DEST)
print(f"Picked: {MBF_TFLITE_SRC} -> {MOBILEFACENET_DEST}")

# Verify shape
it = tf.lite.Interpreter(model_path=MOBILEFACENET_DEST)
it.allocate_tensors()
print("\nMobileFaceNet inputs:")
for t in it.get_input_details():
    print(f"  {t['name']:30s} shape={t['shape'].tolist()} dtype={t['dtype'].__name__}")
print("MobileFaceNet outputs:")
for t in it.get_output_details():
    print(f"  {t['name']:30s} shape={t['shape'].tolist()} dtype={t['dtype'].__name__}")

# Contract: input [1,112,112,3]; output [1, 512]
ishape = it.get_input_details()[0]['shape'].tolist()
oshape = it.get_output_details()[0]['shape'].tolist()
assert ishape == [1, 112, 112, 3], f"MobileFaceNet input {ishape} != [1, 112, 112, 3] — onnx2tf NHWC transpose may have failed"
assert oshape == [1, 512], f"MobileFaceNet output {oshape} != [1, 512]"

## 6 — Functional sanity check

Run a dummy forward pass through each model with a fake image to confirm the interpreter actually runs (not just allocates). And — most importantly — verify that the MobileFaceNet embeddings of two slightly-different inputs are **similar but not identical** (proving the weights are real and not random).

In [ ]:
import numpy as np

def run(model_path, input_np):
    it = tf.lite.Interpreter(model_path=model_path)
    it.allocate_tensors()
    inp = it.get_input_details()[0]
    it.set_tensor(inp['index'], input_np.astype(inp['dtype']))
    it.invoke()
    return [it.get_tensor(o['index']) for o in it.get_output_details()]

# BlazeFace dummy frame [-1, 1]
bf_in = (np.random.rand(1, 128, 128, 3).astype(np.float32) * 2.0) - 1.0
bf_outs = run(BLAZEFACE_DEST, bf_in)
print(f"BlazeFace forward pass ok. Output shapes: {[o.shape for o in bf_outs]}")

# FaceMesh dummy frame [0, 1]
fm_in = np.random.rand(1, 192, 192, 3).astype(np.float32)
fm_outs = run(FACEMESH_DEST, fm_in)
print(f"FaceMesh forward pass ok. Output shapes: {[o.shape for o in fm_outs]}")

# MobileFaceNet: two slightly different inputs — embeddings should be SIMILAR but NOT IDENTICAL
mf_in_a = (np.random.rand(1, 112, 112, 3).astype(np.float32) * 2.0) - 1.0
mf_in_b = mf_in_a + (np.random.randn(*mf_in_a.shape).astype(np.float32) * 0.05)  # small perturbation
emb_a = run(MOBILEFACENET_DEST, mf_in_a)[0][0]   # [512]
emb_b = run(MOBILEFACENET_DEST, mf_in_b)[0][0]   # [512]

# L2-normalize per the contract
emb_a /= np.linalg.norm(emb_a) + 1e-9
emb_b /= np.linalg.norm(emb_b) + 1e-9
cosine_distance = 1.0 - float(np.dot(emb_a, emb_b))

print(f"\nMobileFaceNet embedding norms (pre-L2): a={np.linalg.norm(run(MOBILEFACENET_DEST, mf_in_a)[0][0]):.3f}, b={np.linalg.norm(run(MOBILEFACENET_DEST, mf_in_b)[0][0]):.3f}")
print(f"Cosine distance between perturbed-pair embeddings: {cosine_distance:.4f}")
print("Interpretation: should be > 0 (perturbation moved the embedding) AND < ~0.5 (still close — real weights).")
print("If this prints ~0.0 every time, the model is degenerate. If ~1.0, weights are likely random.")

## 7 — Final summary

These three files in `/kaggle/working/models/` are what you download and commit into `mobile_app/assets/models/` on your laptop.

In [ ]:
print("=" * 78)
print(f"{'File':<28} {'Input shape':<22} {'Output shape(s)':<28} {'Size'}")
print("=" * 78)

total_bytes = 0
for fn in sorted(os.listdir(STAGING)):
    path = os.path.join(STAGING, fn)
    if not fn.endswith(".tflite"):
        continue
    it = tf.lite.Interpreter(model_path=path)
    it.allocate_tensors()
    ishape = it.get_input_details()[0]['shape'].tolist()
    oshapes = [t['shape'].tolist() for t in it.get_output_details()]
    size = os.path.getsize(path)
    total_bytes += size
    print(f"{fn:<28} {str(ishape):<22} {str(oshapes):<28} {size/1024/1024:.2f} MB")

print("=" * 78)
print(f"Total bundle size (excl. ShuffleNet): {total_bytes/1024/1024:.2f} MB")
print(f"Contract cap: 20 MB (models only). Remaining budget after ShuffleNet: ~{20 - total_bytes/1024/1024 - 1.5:.1f} MB (estimating ShuffleNet INT8 at ~1.5 MB).")

## What to send back to Claude

1. **Paste the output of cells 5, 7, 9, 11, 12 verbatim** (the shape print blocks for each model + the functional sanity output + the final summary table).
2. If any cell failed, paste the **full traceback**.

## What to do on your laptop after this runs green

```bash
# Download the three .tflite files from Kaggle. Either via the UI Output panel,
# or via CLI:
kaggle kernels output <your-username>/02-phase1-pretrained-models -p /tmp/kaggle_out

# Move into the repo (NEW filenames — don't delete the dummies in this commit)
cd ~/Desktop/nhai
mkdir -p mobile_app/assets/models
mv /tmp/kaggle_out/models/blazeface.tflite       mobile_app/assets/models/
mv /tmp/kaggle_out/models/facemesh.tflite        mobile_app/assets/models/
mv /tmp/kaggle_out/models/mobilefacenet.tflite   mobile_app/assets/models/

git add mobile_app/assets/models/
git commit -m "ml(phase1): ship real pretrained .tflite (BlazeFace, FaceMesh, MobileFaceNet w600k_mbf)"
git push
```

Then tell teammate: "new models available — point your loader at `blazeface.tflite`, `facemesh.tflite`, `mobilefacenet.tflite`. Dummies are kept for one commit as fallback."

**Do NOT click "Save Version"** — the durable artifacts are the `.tflite` files (already downloaded to the repo); the notebook itself doesn't need a Kaggle snapshot.